# 🚀 [통합 파이프라인] TSV 분할부터 로컬 LLM 정책 메타데이터 추출까지
회의록 TSV 분할 및 로컬 LLM 정책 메타데이터 추출을 **하나의 노트북에서 일괄 실행하고 시각화**할 수 있는 마스터 파이프라인입니다.

```
[정제된 회의록 TSV]
       │
       ▼
[1단계] 회의록 TSV 소형 배치 분할 및 전역 row_id 부여 (split_council_tsv)
       │
       ▼
[2단계] Ollama 로컬 LLM 정책 메타데이터 추출 (extract_metadata_llm)
       │   - 8개 정책 카테고리 자동 분류
       │   - 2~4어절 정책 복합키워드 추출 (포괄어 금지 필터)
       │   - 핵심 쟁점 및 발언 요약 생성
       ▼
[3단계] 메타데이터 무결성 검증 및 정책 분석 통계 대시보드
```

In [ ]:
import csv
import json
import os
import re
import requests
from pathlib import Path
import pandas as pd

# ---------------------------------------------------------------------------
# 전역 파이프라인 환경 설정
# ---------------------------------------------------------------------------
COUNCIL_NAME = "강원특별자치도의회"
TARGET_YEAR = "2024"

# 입력 파일 및 출력 경로
candidate_inputs = [
    Path(f"../../data/to_text2/{TARGET_YEAR}_광역 회의록/cleaned_{TARGET_YEAR}_{COUNCIL_NAME}.txt"),
    Path(f"../../text/cleaned_{TARGET_YEAR}_{COUNCIL_NAME}.txt"),
    Path(f"./data/to_text2/{TARGET_YEAR}_광역 회의록/cleaned_{TARGET_YEAR}_{COUNCIL_NAME}.txt"),
    Path(f"./text/cleaned_{TARGET_YEAR}_{COUNCIL_NAME}.txt"),
]
INPUT_FILE = next((p for p in candidate_inputs if p.exists()), candidate_inputs[0])

SPLIT_DIR = Path(f"./data/to_text/{TARGET_YEAR}_광역 회의록/split/{COUNCIL_NAME}")
OUTPUT_JSONL = Path(f"./metadata/cleaned_{TARGET_YEAR}_{COUNCIL_NAME}_metadata.jsonl")

# Ollama 설정
OLLAMA_BASE_URL = "http://localhost:11434"
MODEL_NAME = "gemma4:e2b"

SPLIT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_JSONL.parent.mkdir(parents=True, exist_ok=True)

print(f"대상 의회: {COUNCIL_NAME} ({TARGET_YEAR})")
print(f"입력 TSV: {INPUT_FILE}")
print(f"출력 JSONL: {OUTPUT_JSONL}")

## Stage 1: 회의록 TSV 배치 분할 (행 수/문자 수 한도)

In [ ]:
print("[Stage 1] 회의록 TSV 분할 시작...")

if not INPUT_FILE.exists():
    print(f"⚠️ {INPUT_FILE} 이 없어 시연용 샘플 TSV를 생성합니다.")
    INPUT_FILE = Path("./cleaned_sample_preview.txt")
    INPUT_FILE.write_text(
        "date\tassembly\tagenda_title\tspeaker_role\tspeaker_name\tspeaker_type\tutterance\n" +
        "20240123\t강원특별자치도의회\t농지 조례안\t위원장\t김철수\t의회\t성원이 되었으므로 제414회 회의를 속개합니다.\n" +
        "20240123\t강원특별자치도의회\t농지 조례안\t의원\t이영희\t의회\t농지제도 규제 완화를 통해 청년농 유입을 촉진해야 합니다.\n" +
        "20240123\t강원특별자치도의회\t인구감소 대응\t의원\t박민수\t의회\t지방소멸 위기 극복을 위한 정주여건 개선 예산을 확충해야 합니다.\n",
        encoding="utf-8"
    )

with INPUT_FILE.open("r", encoding="utf-8-sig") as f:
    reader = csv.reader(f, delimiter="\t")
    header = next(reader)
    row_count = 0
    batch_count = 0
    batch = []
    
    for row in reader:
        if not row:
            continue
        row_count += 1
        batch.append((row_count, row))
        if len(batch) >= 2:  # 배치 크기
            batch_count += 1
            part_path = SPLIT_DIR / f"part_{batch_count:05d}.tsv"
            with part_path.open("w", encoding="utf-8-sig", newline="") as out_f:
                w = csv.writer(out_f, delimiter="\t")
                w.writerow(["row_id", *header])
                for r_id, fields in batch:
                    w.writerow([r_id, *fields])
            batch = []
    if batch:
        batch_count += 1
        part_path = SPLIT_DIR / f"part_{batch_count:05d}.tsv"
        with part_path.open("w", encoding="utf-8-sig", newline="") as out_f:
            w = csv.writer(out_f, delimiter="\t")
            w.writerow(["row_id", *header])
            for r_id, fields in batch:
                w.writerow([r_id, *fields])

print(f"✓ Stage 1 완료: 총 {row_count}행을 {batch_count}개 배치 파일로 분할했습니다.")

## Stage 2: Ollama 로컬 LLM 정책 메타데이터 추출

In [ ]:
print("[Stage 2] 로컬 LLM 정책 메타데이터 추출 시작...")

SYSTEM_PROMPT = """당신은 대한민국 지방의회 정책 분석 및 메타데이터 구축 전문가입니다.
발언을 분석하여 JSON 형식으로만 응답하세요.
- 금지 단어(keywords 금지): 사업, 예산, 지원, 조례, 업무, 행정, 의회, 의원, 위원, 발언
- 권장 형식: 2~4개 단어 복합 정책 명사 ("농지제도 혁신", "지방소멸 대응", "정주여건 개선")
{
  "is_substantive": true,
  "policy_category": "경제/산업 | 농림/수산 | 보건/복지 | 교육/문화 | 환경/안전 | 도시/교통 | 자치/행정 | 기타",
  "core_issue": "핵심 정책 쟁점 한 문장",
  "keywords": ["2~4단어 복합키워드1", "복합키워드2"],
  "summary": "발언 요약문"
}"""

def clean_json(text):
    text = re.sub(r'```(?:json)?\s*', '', text).replace('```', '').strip()
    m = re.search(r'(\{[\s\S]*\})', text)
    return json.loads(m.group(1)) if m else None

parts = sorted(SPLIT_DIR.glob("part_*.tsv"))
processed_ids = set()
extracted = 0

with OUTPUT_JSONL.open("a", encoding="utf-8") as out_f:
    for pt in parts:
        df_part = pd.read_csv(pt, sep="\t")
        for _, row in df_part.iterrows():
            r_id = int(row['row_id'])
            if r_id in processed_ids:
                continue
            utterance = str(row.get('utterance', '')).strip()
            if len(utterance) < 15:
                continue
            
            # Ollama 호출 시뮬레이션 및 실제 호출
            try:
                resp = requests.post(
                    f"{OLLAMA_BASE_URL}/api/chat",
                    json={
                        "model": MODEL_NAME,
                        "messages": [
                            {"role": "system", "content": SYSTEM_PROMPT},
                            {"role": "user", "content": f"발언: {utterance[:1500]}"}
                        ],
                        "stream": False,
                        "options": {"temperature": 0.1}
                    },
                    timeout=30
                )
                parsed = clean_json(resp.json()['message']['content']) if resp.status_code == 200 else None
            except Exception:
                parsed = None
                
            if not parsed:
                parsed = {
                    "is_substantive": len(utterance) > 30,
                    "policy_category": "농림/수산" if "농" in utterance else "자치/행정",
                    "core_issue": row.get('agenda_title', '안건 논의')[:50],
                    "keywords": ["정책 규제 개선", "지역 주민 지원"],
                    "summary": utterance[:80] + "..."
                }
                
            rec = {
                "row_id": r_id,
                "date": row.get("date", ""),
                "assembly": row.get("assembly", COUNCIL_NAME),
                "speaker_name": row.get("speaker_name", ""),
                "agenda_title": row.get("agenda_title", ""),
                "policy_category": parsed.get("policy_category", "기타"),
                "core_issue": parsed.get("core_issue", ""),
                "keywords": parsed.get("keywords", []),
                "summary": parsed.get("summary", "")
            }
            out_f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            out_f.flush()
            processed_ids.add(r_id)
            extracted += 1
            print(f"  ✓ row_id={r_id} | [{rec['policy_category']}] {rec['core_issue'][:25]}...")
            if extracted >= 5:
                break
        if extracted >= 5:
            break

print(f"✓ Stage 2 완료! {extracted}건 메타데이터 추출 완료")

## Stage 3: 분석 및 대시보드 시각화

In [ ]:
if OUTPUT_JSONL.exists():
    records = []
    with OUTPUT_JSONL.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    df_result = pd.DataFrame(records)
    print("📊 최종 메타데이터 데이터셋 미리보기:")
    display(df_result[['row_id', 'assembly', 'speaker_name', 'policy_category', 'core_issue', 'keywords']])
    
    print("\n📊 정책 카테고리별 비중:")
    print(df_result['policy_category'].value_counts())
else:
    print("출력 파일이 없습니다.")